#### Fake News Detection Pipeline using Bi-LSTM and TensorFlow.
Author: Yvan Bonival FEUGANG

In [8]:
# Import all the packages needed for the project and define the configuration variables

import io
import re
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models

# Configuration générale
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
RANDOM_STATE = 42
MAX_VOCAB = 10_000
MAX_LEN = 256
BATCH_SIZE = 64
EPOCHS = 2
EMBEDDING_DIM = 128

In [9]:
# Chargement et Préparation des Données

def load_and_preprocess_data(fake_path: str = "Fake.csv", true_path: str = "True.csv") -> pd.DataFrame:
    fake_df = pd.read_csv(fake_path)
    true_df = pd.read_csv(true_path)

    fake_df["class"] = 0
    true_df["class"] = 1

    df = pd.concat([fake_df, true_df], ignore_index=True)

    # Fusion Titre + Contenu pour un meilleur contexte sémantique
    df["full_text"] = df["title"].fillna("") + " " + df["text"].fillna("")

    # Suppression des colonnes non généralisables
    df = df[["full_text", "class"]].dropna()
    return df


def clean_text(text: str) -> str:
    """Nettoyage regex rapide du texte brut."""
    text = text.lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


print("Chargement des données...")
df = load_and_preprocess_data()
df["full_text"] = df["full_text"].apply(clean_text)

# Partitionnement Train / Test
X_train, X_test, y_train, y_test = train_test_split(
    df["full_text"].values,
    df["class"].values,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=df["class"].values,
)

Chargement des données...


In [10]:
# 2. Vectorisation du Texte (Keras)

vectorizer = layers.TextVectorization(
    max_tokens=MAX_VOCAB,
    output_mode="int",
    output_sequence_length=MAX_LEN,
)
vectorizer.adapt(X_train)

# Création des datasets tf.data
train_ds = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train))
    .shuffle(10_000, seed=RANDOM_STATE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

test_ds = (
    tf.data.Dataset.from_tensor_slices((X_test, y_test))
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

In [11]:
# 3. Architecture du Modèle Récurrent (Bi-LSTM)

def build_model(vocab_size: int = MAX_VOCAB, emb_dim: int = EMBEDDING_DIM) -> tf.keras.Model:
    inputs = layers.Input(shape=(1,), dtype=tf.string, name="input_text")
    x = vectorizer(inputs)
    x = layers.Embedding(input_dim=vocab_size, output_dim=emb_dim, name="embedding")(x)
    x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(x)
    x = layers.Bidirectional(layers.LSTM(32))(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(1, activation="sigmoid", name="output_probability")(x)

    model = models.Model(inputs=inputs, outputs=outputs, name="FakeNews_BiLSTM")
    return model


model = build_model()
model.summary()

model.compile(
    loss="binary_crossentropy",
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    metrics=["accuracy"],
)

Model: "FakeNews_BiLSTM"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_text (InputLayer)         │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization_1            │ (None, 256)            │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 256, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 256, 128)       │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ (None, 64)             │        41,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_probability (Dense)      │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,424,257 (5.43 MB)

 Trainable params: 1,424,257 (5.43 MB)

 Non-trainable params: 0 (0.00 B)

In [12]:
# 4. Entraînement

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True,
    verbose=1,
)

history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=EPOCHS,
    callbacks=[early_stop],
)

Epoch 1/2


c:\Users\boniv\Downloads\Github Project\Fake-news-dectection-using-Recurrent-Neral-Network\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


562/562 ━━━━━━━━━━━━━━━━━━━━ 371s 647ms/step - accuracy: 0.9441 - loss: 0.1683 - val_accuracy: 0.9982 - val_loss: 0.0090
Epoch 2/2
562/562 ━━━━━━━━━━━━━━━━━━━━ 365s 650ms/step - accuracy: 0.9989 - loss: 0.0084 - val_accuracy: 0.9986 - val_loss: 0.0061
Restoring model weights from the end of the best epoch: 2.


In [13]:
# 5. Évaluation et Métriques

y_pred_probs = model.predict(test_ds).ravel()
y_pred = (y_pred_probs >= 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)

print(f"\n--- RÉSULTATS DE TEST ---")
print(f"Accuracy : {acc * 100:.2f}%")
print(f"Precision: {prec * 100:.2f}%")
print(f"Recall   : {rec * 100:.2f}%\n")
print(classification_report(y_test, y_pred, target_names=["Fake", "Real"]))

# Matrice de confusion normalisée
cm = confusion_matrix(y_test, y_pred, normalize="true")
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt=".2%", cmap="Blues", xticklabels=["Fake", "Real"], yticklabels=["Fake", "Real"])
plt.title("Matrice de Confusion Normalisée")
plt.xlabel("Prédictions")
plt.ylabel("Vérité terrain")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=300)
plt.close()

141/141 ━━━━━━━━━━━━━━━━━━━━ 23s 154ms/step

--- RÉSULTATS DE TEST ---
Accuracy : 99.86%
Precision: 99.88%
Recall   : 99.81%

              precision    recall  f1-score   support

        Fake       1.00      1.00      1.00      4696
        Real       1.00      1.00      1.00      4284

    accuracy                           1.00      8980
   macro avg       1.00      1.00      1.00      8980
weighted avg       1.00      1.00      1.00      8980



In [14]:
# 6. Export pour Tensorflow Embedding Projector

def export_embeddings(model, vectorizer, filename_vec="vecs.tsv", filename_meta="meta.tsv"):
    embedding_layer = model.get_layer("embedding")
    weights = embedding_layer.get_weights()[0]
    vocab = vectorizer.get_vocabulary()

    with io.open(filename_vec, "w", encoding="utf-8") as f_vec, io.open(
        filename_meta, "w", encoding="utf-8"
    ) as f_meta:
        for index, word in enumerate(vocab):
            if index == 0:
                continue  # ignore le token de padding
            vec = weights[index]
            f_meta.write(word + "\n")
            f_vec.write("\t".join([str(x) for x in vec]) + "\n")

    print(f"Embeddings exportés dans {filename_vec} et {filename_meta}")


export_embeddings(model, vectorizer)

Embeddings exportés dans vecs.tsv et meta.tsv
